# 使用 `ov.llm` 调用基础模型

`ov.llm.ModelFactory` 是单细胞基础模型的统一入口。`create_model(model_type, model_path, device, **kwargs)` 返回模型对象（`ScGPTModel`、`ScFoundationModel`、`GeneformerModel`、`CellPLMModel`、`UCEModel`、`ScMulanModel` 等），由其方法（`get_embeddings`、`predict`、`fine_tune`、`integrate` 等）完成具体工作。

本 notebook 概览这些入口。前几个单元格不需要任何权重，已执行并保留输出；从“创建模型”开始的单元格需要模型权重（和 GPU），**此处未执行**，对应的已执行版本见各模型教程（scGPT、scFoundation、Geneformer、CellPLM、UCE）。

## 已注册的 model type

In [1]:
import omicverse as ov

model_types = ov.llm.ModelFactory.available_models()
print(len(model_types), model_types)

22 ['scgpt', 'scfoundation', 'geneformer', 'cellplm', 'uce', 'scmulan', 'scbert', 'genecompass', 'nicheformer', 'tgpt', 'cellfm', 'sccello', 'scprint', 'aidocell', 'pulsar', 'atacformer', 'scplantllm', 'langcell', 'cell2sentence', 'genept', 'chatcell', 'tabula']


其中 6 个已集成，其余 16 个是 `omicverse.llm.external_model_stubs` 中的占位类。

In [2]:
from omicverse.llm.external_model_stubs import ExternalModelStub

factory = ov.llm.ModelFactory
integrated = [k for k in model_types if not issubclass(factory._models[k], ExternalModelStub)]
placeholders = [k for k in model_types if k not in integrated]
print('integrated :', integrated)
print('placeholder:', placeholders)

integrated : ['scgpt', 'scfoundation', 'geneformer', 'cellplm', 'uce', 'scmulan']
placeholder: ['scbert', 'genecompass', 'nicheformer', 'tgpt', 'cellfm', 'sccello', 'scprint', 'aidocell', 'pulsar', 'atacformer', 'scplantllm', 'langcell', 'cell2sentence', 'genept', 'chatcell', 'tabula']


## 占位模型

占位模型无需权重即可创建，但所有任务都会抛出带原因和上游地址的 `ModelExecutionBlockedError`，不会计算任何结果。

In [3]:
import anndata

model = ov.llm.ModelFactory.create_model('scbert')
print(model)
try:
    model.get_embeddings(anndata.AnnData())   # raises before touching the data
except ov.llm.ModelExecutionBlockedError as err:
    print(err)
    print(err.checkpoint_url)

ScBERTModel(model_name=scbert, device=cpu, status=not loaded)
scbert cannot run task 'embed': scBERT is registered but the upstream inference runtime is not yet integrated into omicverse.llm.
https://github.com/TencentAILabHealthcare/scBERT


## 创建模型（需要权重）

所有已集成模型的创建方式一致。权重来源见各模型页面以及 `omicverse.llm.model_download`。

In [ ]:
model = ov.llm.ModelFactory.create_model(
    model_type="scgpt",
    model_path="llm_model/models/scgpt",
    device="cuda",
)
embeddings = model.get_embeddings(adata, skip_normalization=True)   # (n_cells, 512)

### 注释与整合

`fine_tune` 在带标签的参考集上微调；`predict_celltypes`（scGPT、scFoundation、Geneformer、CellPLM）或 `predict(task="annotation")` 为查询集注释；`integrate` 在嵌入之上做批次校正。

In [ ]:
model.fine_tune(train_adata=reference_adata, epochs=5, batch_size=32, lr=1e-4,
                mask_ratio=0.0, data_is_raw=False, log1p=False, skip_normalization=True)
res = model.predict_celltypes(adata, skip_normalization=True)
adata.obs["predicted_celltype"] = res["predicted_celltypes"]
res = model.integrate(adata, batch_key="batch", correction_method="mnn",
                      log1p=False, data_is_raw=False, skip_normalization=True)
adata.obsm["X_scgpt_fine"] = res["embeddings"]

## 其他已集成模型

scFoundation 与 CellPLM 传入权重路径；Geneformer 先创建，再用三个字典文件加载；UCE 需要五个资源路径；scMulan 仅支持嵌入与零样本注释。

In [ ]:
sf = ov.llm.ModelFactory.create_model("scfoundation", "llm_model/models/scfoundation/models.ckpt", "cuda")
emb = sf.get_embeddings(adata, pre_normalized="T", input_type="singlecell")   # (n_cells, 3072)
cp = ov.llm.ModelFactory.create_model("cellplm", "llm_model/models/cellplm", "cuda",
                                      pretrain_version="20231027_85M")
emb = cp.get_embeddings(adata)   # (n_cells, 512)

In [ ]:
gf = ov.llm.ModelFactory.create_model("geneformer", device="cuda", model_version="V2")
gf.load_model("llm_model/models/geneformer/Geneformer-V2-104M",
              gene_median_file="llm_model/models/geneformer/gene_median_dictionary_gc104M.pkl",
              token_dictionary_file="llm_model/models/geneformer/token_dictionary_gc104M.pkl",
              gene_mapping_file="llm_model/models/geneformer/ensembl_mapping_dict_gc104M.pkl")
emb = gf.get_embeddings(adata, max_ncells=100000)   # (n_cells, 768)

In [ ]:
uce = ov.llm.ModelFactory.create_model(
    "uce", "llm_model/models/UCE/4layer_model.torch", "cuda", species="human",
    token_file="llm_model/models/UCE/all_tokens.torch",
    protein_embeddings_dir="llm_model/models/UCE/protein_embeddings/",
    spec_chrom_csv_path="llm_model/models/UCE/species_chrom.csv",
    offset_pkl_path="llm_model/models/UCE/species_offsets.pkl")
emb = uce.get_embeddings(adata, filter_cells=False, filter_genes=False)   # (n_cells, 1280)

In [ ]:
mulan = ov.llm.ModelFactory.create_model("scmulan", "llm_model/models/scmulan/ckpt_scMulan.pt", "cuda")
emb = mulan.get_embeddings(adata)
res = mulan.predict(adata, task="annotation")   # res["predicted_celltypes"], res["embeddings"]

## 各模型的辅助函数

`omicverse.llm.model_factory` 为 scGPT、scFoundation、Geneformer、CellPLM 提供“创建 + 运行”一步到位的函数（如 `get_embeddings_with_cellplm`、`end_to_end_scgpt_annotation`、`perturb_genes_with_geneformer`）。UCE 与 scMulan 没有。完整列表与签名见各模型页面。

In [ ]:
from omicverse.llm.model_factory import get_embeddings_with_cellplm

emb = get_embeddings_with_cellplm(adata, model_path="llm_model/models/cellplm",
                                  pretrain_version="20231027_85M", device="cuda")

## 从已移除的管理器类迁移

`manager = Manager(model_type=T, model_path=P, device=D)` 改为 `model = ov.llm.ModelFactory.create_model(T, P, D)`；`manager.model.<method>` 改为 `model.<method>`；`manager.annotate_cells(adata)` 改为 `model.predict(adata, task="annotation")`；`manager.get_embeddings`、`integrate`、`fine_tune`、`train_integration`、`save_model` 在模型对象上保持同名。

另有 16 个 model type 仅为占位，见“尚未集成”页面。